# Domácí úkol 1S — Explorační analýza dat FPS
**Předmět:** Pravděpodobnost a statistika  
**Téma:** Vliv patche 1.5 ve hře *Cyberpunk 2077* na výkonnostní skóre (FPS) grafických karet  
**Zkoumané grafické karty:** *Nvidia RTX 3070 Ti* a *AMD Radeon RX 6800 XT*  

---
### Popis úlohy a cíl analýzy:
Cílem práce je provést explorační analýzu nárůstu FPS po aplikaci patche 1.5 na testovaných sestavách, tedy párového rozdílu:
$$\Delta \text{FPS} = \text{FPS}_{\text{patched}} - \text{FPS}_{\text{release}}$$

**Postup práce:**
1. Načtení a příprava dat (import, výpočet nárůstu FPS, převod do dlouhého formátu).
2. Vícenásobný krabicový graf pro původní data.
3. Identifikace a vyřazení odlehlých pozorování pomocí Tukeyho vnitřních hradeb.
4. Vícenásobný krabicový graf po odstranění odlehlých pozorování.
5. Vizuální posouzení normality (histogramy s odhady hustot a Q-Q grafy).
6. Souhrnná tabulka popisných statistik.
7. Závěrečné vyhodnocení (posouzení normality a srovnání výkonu karet).


## 1. Načtení a příprava dat


In [ ]:
# Nastavení prostředí a načtení knihoven
options(repr.plot.width = 8.5, repr.plot.height = 5.2, repr.plot.res = 120)
pdf.options(bg = "white")
set.seed(123)

library(dplyr)
library(tidyr)
library(readxl)
library(rstatix)
library(moments)


In [ ]:
# Načtení dat ze souboru FPS_data.xlsx
cesta_dat <- if (file.exists("FPS_data.xlsx")) "FPS_data.xlsx" else file.path("du1", "FPS_data.xlsx")
stopifnot(file.exists(cesta_dat))

data_raw <- read_excel(cesta_dat, sheet = "Vysledky mereni")
head(data_raw)


In [ ]:
# Přejmenování sloupců pro přehlednou syntaxi
data_clean <- data_raw %>%
  rename(
    id = id,
    rtx2080ti_rel = `Nvidia RTX 2080 Ti average FPS release (FPS)`,
    rtx2080ti_pat = `Nvidia RTX 2080 Ti average FPS patched (FPS)`,
    rtx3070ti_rel = `Nvidia RTX 3070 Ti average FPS release (FPS)`,
    rtx3070ti_pat = `Nvidia RTX 3070 Ti average FPS patched (FPS)`,
    rx6800xt_rel = `AMD Radeon RX 6800 XT average FPS release (FPS)`,
    rx6800xt_pat = `AMD Radeon RX 6800 XT average FPS patched (FPS)`,
    rx7700xt_rel = `AMD Radeon RX 7700 XT average FPS release (FPS)`,
    rx7700xt_pat = `AMD Radeon RX 7700 XT average FPS patched (FPS)`
  )

# Výpočet párového rozdílu FPS a převod do dlouhého formátu (tidy data)
data_diff <- data_clean %>%
  mutate(
    narust_rtx3070ti = rtx3070ti_pat - rtx3070ti_rel,
    narust_rx6800xt = rx6800xt_pat - rx6800xt_rel
  )

data_long <- data_diff %>%
  select(id, narust_rtx3070ti, narust_rx6800xt) %>%
  pivot_longer(
    cols = c(narust_rtx3070ti, narust_rx6800xt),
    names_to = "karta",
    values_to = "narust"
  ) %>%
  mutate(
    row_id = row_number(),
    karta = factor(case_when(
      karta == "narust_rtx3070ti" ~ "Nvidia RTX 3070 Ti",
      karta == "narust_rx6800xt" ~ "AMD Radeon RX 6800 XT"
    ), levels = c("Nvidia RTX 3070 Ti", "AMD Radeon RX 6800 XT"))
  )

head(data_long, 6)


## 2. Vícenásobný krabicový graf pro původní data


In [ ]:
par(mar = c(4.5, 4.5, 3.5, 1.5))
boxplot(narust ~ karta, data = data_long,
        xlab = "Grafická karta",
        ylab = "Nárůst výkonu [FPS]",
        main = "Obr. 1: Vícenásobný krabicový graf nárůstu FPS pro původní data",
        cex.main = 1.05,
        cex.lab = 1.0,
        col = c("#74a9cf", "#fd8d3c"),
        border = c("#02818a", "#e6550d"))
grid(nx = NA, ny = NULL)


### Komentář k Obr. 1:
V původních datech se vyskytují výrazná odlehlá pozorování:
- U **Nvidia RTX 3070 Ti** propad na $-5{,}50$ FPS (extrémní záporná hodnota pod dolním vousem).
- U **AMD Radeon RX 6800 XT** extrémní zisk $+13{,}10$ FPS a mírný zisk $+7{,}50$ FPS nad horním vousem.
- Vlivem těchto bodů je osa Y roztažena na $[-6, 14]$ FPS a samotné krabice (IQR) jsou špatně čitelné.


## 3. Identifikace a vyřazení odlehlých pozorování


In [ ]:
# Detekce odlehlých pozorování pomocí Tukeyho vnitřních hradeb [Q1 - 1.5*IQR, Q3 + 1.5*IQR]
odlehla_pozorovani <- data_long %>%
  group_by(karta) %>%
  rstatix::identify_outliers(narust)

odlehla_pozorovani


In [ ]:
# Vytvoření proměnné bez odlehlých pozorování (nahrazení NA)
data_long <- data_long %>%
  mutate(narust_bez_op = ifelse(row_id %in% odlehla_pozorovani$row_id, NA, narust))

# Přehled počtu platných pozorování
data_long %>%
  group_by(karta) %>%
  summarise(
    celkem = n(),
    vyrazenych_OP = sum(row_id %in% odlehla_pozorovani$row_id),
    platnych_ocistenych = sum(!is.na(narust_bez_op))
  )


### Shrnutí odlehlých pozorování:
- **Nvidia RTX 3070 Ti:** `id = 38` ($-5{,}50$ FPS, extrémní propad).
- **AMD Radeon RX 6800 XT:** `id = 7` ($+13{,}10$ FPS, extrémní zisk), `id = 15` ($+7{,}50$ FPS, mírně odlehlé).
- Tyto 3 hodnoty byly nahrazeny `NA` v proměnné `narust_bez_op`.


## 4. Vícenásobný krabicový graf po odstranění odlehlých pozorování


In [ ]:
par(mar = c(4.5, 4.5, 3.5, 1.5))
boxplot(narust_bez_op ~ karta, data = data_long,
        xlab = "Grafická karta",
        ylab = "Nárůst výkonu [FPS]",
        main = "Obr. 2: Vícenásobný krabicový graf nárůstu FPS (po odstranění odlehlých pozorování)",
        cex.main = 1.05,
        cex.lab = 1.0,
        col = c("#74a9cf", "#fd8d3c"),
        border = c("#02818a", "#e6550d"))
grid(nx = NA, ny = NULL)


### Komentář k Obr. 2:
- Po vyřazení odlehlých bodů se rozsah osy Y zúžil z $[-6, 14]$ na cca $[3, 7]$ FPS (přibližně 5× větší detail).
- Medián nárůstu výkonu u **Nvidia RTX 3070 Ti** je $5{,}20$ FPS, u **AMD Radeon RX 6800 XT** je $4{,}70$ FPS.
- Variabilita ($IQR$) je u obou karet velmi nízká a srovnatelná ($0{,}80$ resp. $0{,}85$ FPS).


## 5. Vizuální posouzení normality (Histogramy a Q-Q grafy)


In [ ]:
fps_rtx_clean <- data_long %>%
  filter(karta == "Nvidia RTX 3070 Ti", !is.na(narust_bez_op)) %>%
  pull(narust_bez_op)

fps_rx_clean <- data_long %>%
  filter(karta == "AMD Radeon RX 6800 XT", !is.na(narust_bez_op)) %>%
  pull(narust_bez_op)

# Sjednocený rozsah osy X pro oba histogramy (zajistí popisky od 3 do 7 a přímou srovnatelnost)
xlim_spolecne <- c(2.7, 7.1)

# --- Histogram Nvidia RTX 3070 Ti ---
par(mar = c(4.5, 4.5, 3.8, 1.5))
dens_rtx <- density(fps_rtx_clean)
max_h_rtx <- max(hist(fps_rtx_clean, breaks = 10, plot = FALSE)$density, dens_rtx$y, dnorm(mean(fps_rtx_clean), mean(fps_rtx_clean), sd(fps_rtx_clean)))
ylim_rtx <- c(0, max_h_rtx * 1.35)

hist(fps_rtx_clean, freq = FALSE, breaks = 10, xlim = xlim_spolecne, ylim = ylim_rtx,
     xlab = "Nárůst výkonu [FPS]", ylab = "Hustota pravděpodobnosti",
     main = "Obr. 3: Histogram nárůstu FPS pro Nvidia RTX 3070 Ti\n(po odstranění odlehlých pozorování)",
     cex.main = 1.0, cex.lab = 1.0, col = "#c6dbef", border = "#08519c")
grid()
lines(dens_rtx, col = "red", lwd = 2)
osa_x_rtx <- seq(xlim_spolecne[1], xlim_spolecne[2], length.out = 200)
lines(osa_x_rtx, dnorm(osa_x_rtx, mean = mean(fps_rtx_clean), sd = sd(fps_rtx_clean)), col = "blue", lwd = 2, lty = 2)
legend("topright", legend = c("Jádrový odhad hustoty", "Teoretické normální rozdělení"),
       col = c("red", "blue"), lty = c(1, 2), lwd = 2, cex = 0.85, bg = "white", box.col = "#bdbdbd")


In [ ]:
# --- Histogram AMD Radeon RX 6800 XT ---
par(mar = c(4.5, 4.5, 3.8, 1.5))
dens_rx <- density(fps_rx_clean)
max_h_rx <- max(hist(fps_rx_clean, breaks = 10, plot = FALSE)$density, dens_rx$y, dnorm(mean(fps_rx_clean), mean(fps_rx_clean), sd(fps_rx_clean)))
ylim_rx <- c(0, max_h_rx * 1.35)

hist(fps_rx_clean, freq = FALSE, breaks = 10, xlim = xlim_spolecne, ylim = ylim_rx,
     xlab = "Nárůst výkonu [FPS]", ylab = "Hustota pravděpodobnosti",
     main = "Obr. 4: Histogram nárůstu FPS pro AMD Radeon RX 6800 XT\n(po odstranění odlehlých pozorování)",
     cex.main = 1.0, cex.lab = 1.0, col = "#fdd0a2", border = "#a63603")
grid()
lines(dens_rx, col = "red", lwd = 2)
osa_x_rx <- seq(xlim_spolecne[1], xlim_spolecne[2], length.out = 200)
lines(osa_x_rx, dnorm(osa_x_rx, mean = mean(fps_rx_clean), sd = sd(fps_rx_clean)), col = "blue", lwd = 2, lty = 2)
legend("topright", legend = c("Jádrový odhad hustoty", "Teoretické normální rozdělení"),
       col = c("red", "blue"), lty = c(1, 2), lwd = 2, cex = 0.85, bg = "white", box.col = "#bdbdbd")


In [ ]:
# --- Q-Q grafy ---
par(mar = c(4.5, 4.5, 3.8, 1.5))
qq_range_rtx <- range(fps_rtx_clean)
qqnorm(fps_rtx_clean,
       xlab = "Teoretické kvantily [-]", ylab = "Výběrové kvantily [FPS]",
       main = "Obr. 5: Q-Q graf nárůstu FPS pro Nvidia RTX 3070 Ti\n(po odstranění odlehlých pozorování)",
       ylim = c(qq_range_rtx[1] - 0.4, qq_range_rtx[2] + 0.4), pch = 20, col = "#08519c", cex = 1.2)
qqline(fps_rtx_clean, col = "red", lwd = 2)
grid()

qq_range_rx <- range(fps_rx_clean)
qqnorm(fps_rx_clean,
       xlab = "Teoretické kvantily [-]", ylab = "Výběrové kvantily [FPS]",
       main = "Obr. 6: Q-Q graf nárůstu FPS pro AMD Radeon RX 6800 XT\n(po odstranění odlehlých pozorování)",
       ylim = c(qq_range_rx[1] - 0.4, qq_range_rx[2] + 0.4), pch = 20, col = "#a63603", cex = 1.2)
qqline(fps_rx_clean, col = "red", lwd = 2)
grid()


### Komentář k histogramům a Q-Q grafům:
- **Histogramy (Obr. 3 a 4):** Mají relativně symetrický zvonovitý tvar s jedním vrcholem. Červená křivka (jádrový odhad hustoty) z naměřených dat se téměř přesně překrývá s modrou křivkou (teoretické normální rozdělení).
- **Q-Q grafy (Obr. 5 a 6):** Všechny body leží těsně podél červené přímky (teoretická přímka normality) a nikde na koncích neuhýbají. Grafy jasně potvrzují normální rozdělení.


## 6. Souhrnná tabulka popisných statistik


In [ ]:
spocti_statistiky <- function(df, sloupec, nazev_stavu) {
  df %>%
    group_by(karta) %>%
    summarise(
      Stav = nazev_stavu,
      n = sum(!is.na({{ sloupec }})),
      Prumer = round(mean({{ sloupec }}, na.rm = TRUE), 3),
      Sm_Odchylka = round(sd({{ sloupec }}, na.rm = TRUE), 3),
      Median = round(median({{ sloupec }}, na.rm = TRUE), 3),
      IQR = round(IQR({{ sloupec }}, na.rm = TRUE), 3),
      Min = round(min({{ sloupec }}, na.rm = TRUE), 3),
      Max = round(max({{ sloupec }}, na.rm = TRUE), 3),
      Sikmost = round(moments::skewness({{ sloupec }}, na.rm = TRUE), 3),
      Spicatost = round(moments::kurtosis({{ sloupec }}, na.rm = TRUE) - 3, 3),
      .groups = "drop"
    )
}

tab_puvodni <- spocti_statistiky(data_long, narust, "Původní data")
tab_ocistena <- spocti_statistiky(data_long, narust_bez_op, "Bez odlehlých pozorování")

tabulka_vysledku <- bind_rows(tab_puvodni, tab_ocistena) %>%
  arrange(karta, desc(Stav))

tabulka_vysledku


## 7. Závěrečné vyhodnocení

### 1. Posouzení normality:
- **Původní data:** Vlivem odlehlých pozorování vykazovala extrémní hodnoty šikmosti a špičatosti (RTX 3070 Ti: šikmost $-6{,}09$, špičatost $43{,}60$; RX 6800 XT: $+4{,}33$, $26{,}50$), data tedy nebyla normálně rozdělená.
- **Očištěná data:** Výběrová šikmost i špičatost klesly hluboko do referenčního intervalu $(-2, 2)$:
  - *Nvidia RTX 3070 Ti:* šikmost $= 0{,}131$, špičatost $= -0{,}220$.
  - *AMD Radeon RX 6800 XT:* šikmost $= 0{,}223$, špičatost $= -0{,}182$.
- Spolu s histogramy a Q-Q grafy lze říci, že **očištěná data velmi dobře odpovídají normálnímu rozdělení**.

---

### 2. Zhodnocení nárůstu výkonu (Cyberpunk 2077, patch 1.5):
1. **Jednoznačné zlepšení výkonu:** Obě karty zaznamenaly statisticky i prakticky prokazatelný nárůst průměrného FPS (všechna očištěná měření leží v kladném intervalu od $3{,}2$ do $6{,}6$ FPS).
2. **Srovnání karet:**
   - **Nvidia RTX 3070 Ti** dosáhla průměrného nárůstu **$5{,}28$ FPS** (medián $5{,}20$ FPS, směrodatná odchylka $0{,}59$ FPS).
   - **AMD Radeon RX 6800 XT** dosáhla průměrného nárůstu **$4{,}75$ FPS** (medián $4{,}70$ FPS, směrodatná odchylka $0{,}67$ FPS).
   - Karta **Nvidia RTX 3070 Ti profitovala z patche 1.5 v průměru o $0{,}54$ FPS více** než AMD Radeon RX 6800 XT.
   - Variabilita ($s \approx 0{,}6$ až $0{,}7$ FPS) byla u obou modelů srovnatelně nízká, což potvrzuje vysokou stabilitu přínosu patche napříč PC sestavami.
3. **Anomálie sestav:** Jeden systém s kartou Nvidia (ID 38) vykázal propad $-5{,}5$ FPS a jeden systém s kartou AMD (ID 7) extrémní zisk $+13{,}1$ FPS.


## Export grafických výstupů do PDF


In [ ]:
pdf_vystup <- if (dir.exists("du1")) file.path("du1", "graficke_vystupy.pdf") else "graficke_vystupy.pdf"
pdf(pdf_vystup, width = 8.5, height = 5.2, bg = "white")

# 1. Krabicový graf původní
par(mar = c(4.5, 4.5, 3.5, 1.5))
boxplot(narust ~ karta, data = data_long,
        xlab = "Grafická karta", ylab = "Nárůst výkonu [FPS]",
        main = "Obr. 1: Vícenásobný krabicový graf nárůstu FPS pro původní data",
        cex.main = 1.05, cex.lab = 1.0, col = c("#74a9cf", "#fd8d3c"), border = c("#02818a", "#e6550d"))
grid(nx = NA, ny = NULL)

# 2. Krabicový graf očištěný
boxplot(narust_bez_op ~ karta, data = data_long,
        xlab = "Grafická karta", ylab = "Nárůst výkonu [FPS]",
        main = "Obr. 2: Vícenásobný krabicový graf nárůstu FPS (po odstranění odlehlých pozorování)",
        cex.main = 1.05, cex.lab = 1.0, col = c("#74a9cf", "#fd8d3c"), border = c("#02818a", "#e6550d"))
grid(nx = NA, ny = NULL)

# 3. Histogram Nvidia RTX 3070 Ti
dens_rtx <- density(fps_rtx_clean)
max_h_rtx <- max(hist(fps_rtx_clean, breaks = 10, plot = FALSE)$density, dens_rtx$y, dnorm(mean(fps_rtx_clean), mean(fps_rtx_clean), sd(fps_rtx_clean)))
ylim_rtx <- c(0, max_h_rtx * 1.35)

hist(fps_rtx_clean, freq = FALSE, breaks = 10, xlim = xlim_spolecne, ylim = ylim_rtx,
     xlab = "Nárůst výkonu [FPS]", ylab = "Hustota pravděpodobnosti",
     main = "Obr. 3: Histogram nárůstu FPS pro Nvidia RTX 3070 Ti\n(po odstranění odlehlých pozorování)",
     cex.main = 1.0, cex.lab = 1.0, col = "#c6dbef", border = "#08519c")
grid()
lines(dens_rtx, col = "red", lwd = 2)
osa_x_rtx <- seq(xlim_spolecne[1], xlim_spolecne[2], length.out = 200)
lines(osa_x_rtx, dnorm(osa_x_rtx, mean = mean(fps_rtx_clean), sd = sd(fps_rtx_clean)), col = "blue", lwd = 2, lty = 2)
legend("topright", legend = c("Jádrový odhad hustoty", "Teoretické normální rozdělení"),
       col = c("red", "blue"), lty = c(1, 2), lwd = 2, cex = 0.85, bg = "white", box.col = "#bdbdbd")

# 4. Histogram AMD Radeon RX 6800 XT
dens_rx <- density(fps_rx_clean)
max_h_rx <- max(hist(fps_rx_clean, breaks = 10, plot = FALSE)$density, dens_rx$y, dnorm(mean(fps_rx_clean), mean(fps_rx_clean), sd(fps_rx_clean)))
ylim_rx <- c(0, max_h_rx * 1.35)

hist(fps_rx_clean, freq = FALSE, breaks = 10, xlim = xlim_spolecne, ylim = ylim_rx,
     xlab = "Nárůst výkonu [FPS]", ylab = "Hustota pravděpodobnosti",
     main = "Obr. 4: Histogram nárůstu FPS pro AMD Radeon RX 6800 XT\n(po odstranění odlehlých pozorování)",
     cex.main = 1.0, cex.lab = 1.0, col = "#fdd0a2", border = "#a63603")
grid()
lines(dens_rx, col = "red", lwd = 2)
osa_x_rx <- seq(xlim_spolecne[1], xlim_spolecne[2], length.out = 200)
lines(osa_x_rx, dnorm(osa_x_rx, mean = mean(fps_rx_clean), sd = sd(fps_rx_clean)), col = "blue", lwd = 2, lty = 2)
legend("topright", legend = c("Jádrový odhad hustoty", "Teoretické normální rozdělení"),
       col = c("red", "blue"), lty = c(1, 2), lwd = 2, cex = 0.85, bg = "white", box.col = "#bdbdbd")

# 5. Q-Q graf Nvidia RTX 3070 Ti
qqnorm(fps_rtx_clean, xlab = "Teoretické kvantily [-]", ylab = "Výběrové kvantily [FPS]",
       main = "Obr. 5: Q-Q graf nárůstu FPS pro Nvidia RTX 3070 Ti\n(po odstranění odlehlých pozorování)",
       ylim = c(qq_range_rtx[1] - 0.4, qq_range_rtx[2] + 0.4), pch = 20, col = "#08519c", cex = 1.2)
qqline(fps_rtx_clean, col = "red", lwd = 2)
grid()

# 6. Q-Q graf AMD Radeon RX 6800 XT
qqnorm(fps_rx_clean, xlab = "Teoretické kvantily [-]", ylab = "Výběrové kvantily [FPS]",
       main = "Obr. 6: Q-Q graf nárůstu FPS pro AMD Radeon RX 6800 XT\n(po odstranění odlehlých pozorování)",
       ylim = c(qq_range_rx[1] - 0.4, qq_range_rx[2] + 0.4), pch = 20, col = qqline(fps_rx_clean, col = "red", lwd = 2)
grid()

dev.off()
cat("Soubor graficke_vystupy.pdf byl úspěšně vygenerován se 6 stranami grafů.\n")
